### 변수와 기본 자료형
##### 숫자(int, float)

In [ ]:
age = 42
premium = 5.4
rate = 0.012

##### 문자열(str)

In [ ]:
name = "홍길동"
product = "자동차보험"

##### 리스트(list)

In [ ]:
ages = [42, 35, 28, 55]
ages.append(60)
ages[0]
print(ages[0])
len(ages)

##### 딕셔너리(dict)

In [ ]:
customer = {"name": "홍길동", "age": 42}
customer["name"]
customer.keys()
customer.items()

### 조건문과 반복문
##### if/for/while

In [ ]:
def grade(age: int) -> str:
    if age < 30:
        return "A"     # 청년
    elif age < 50:
        return "B"     # 중장년
    elif age < 65:
        return "C"     # 시니어
    else:
        return "D"     # 고령

print(grade(42))   # → B


In [ ]:
ages = [42, 35, 28, 55, 68]
for a in ages:
    print(a, grade(a))

# while : 조건 만족까지 반복
total = 0
n = 0
while n < len(ages):
    total += ages[n]
    n += 1
print("평균 연령:", total / n)

### 함수 정의와 모듈 활용

##### Functions & Modules

In [ ]:
def calc_premium(age: int, gender: str, product: str) -> float:
    base = {"건강보험": 3, "자동차보험": 8, "생명보험": 5}[product]
    age_factor = 1.0 + max(0, age - 30) * 0.02     # 30세 이상 증가
    gender_factor = 0.95 if gender == "F" else 1.0  # 여성 5% 할인
    return round(base * age_factor * gender_factor, 1)

print(calc_premium(42, "M", "자동차보험"))   # → 9.9

### 파일 입출력과 예외 처리

##### File I/O & try/except

In [ ]:
import csv
from pathlib import Path

def load_csv(path: str) -> list[dict]:
    rows = []
    with open(path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows

data = load_csv("data/customers.csv")
print(len(data), "명 로드됨")


In [ ]:
for row in data:
    try:
        age = int(row["age"])
        premium = calc_premium(
            age, row["gender"], row["product"]
        )
    except KeyError as e:
        print("필드 누락:", e)
    except ValueError as e:
        print("숫자 변환 실패:", row, e)
    else:
        print(row["name"], premium)


### 실습 시나리오

In [ ]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class Customer:
    name: str
    age: int
    gender: Literal["M", "F"]
    product: str        # 건강보험 | 자동차보험 | 생명보험
    coverage: int       # 보장금액 (만원)

def calc_premium(c: Customer) -> float:
    base = {"건강보험": 3, "자동차보험": 8, "생명보험": 5}[c.product]
    age_factor = 1.0 + max(0, c.age - 30) * 0.02
    gender_factor = 0.95 if c.gender == "F" else 1.0
    coverage_factor = c.coverage / 1000
    return round(base * age_factor * gender_factor * coverage_factor, 1)


In [ ]:
import csv
from pathlib import Path
from customer import Customer, calc_premium

def load_customers(path: str) -> list[Customer]:
    customers = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try:
                customers.append(Customer(
                    name=row["name"],
                    age=int(row["age"]),
                    gender=row["gender"],
                    product=row["product"],
                    coverage=int(row["coverage"])
                ))
            except (KeyError, ValueError) as e:
                print("[경고] 건너뜀:", row, e)
    return customers


In [ ]:
customer_code = '''from dataclasses import dataclass
from typing import Literal

@dataclass
class Customer:
    name: str
    age: int
    gender: Literal["M", "F"]
    product: str        # 건강보험 | 자동차보험 | 생명보험
    coverage: int       # 보장금액 (만원)

def calc_premium(c: Customer) -> float:
    base = {
        "건강보험": 3,
        "자동차보험": 8,
        "생명보험": 5
    }[c.product]

    age_factor = 1.0 + max(0, c.age - 30) * 0.02
    gender_factor = 0.95 if c.gender == "F" else 1.0
    coverage_factor = c.coverage / 1000

    return round(base * age_factor * gender_factor * coverage_factor, 1)
'''

with open("customer.py", "w", encoding="utf-8") as f:
    f.write(customer_code)

print("customer.py 파일이 생성되었습니다.")

In [ ]:
import csv
from pathlib import Path
from customer import Customer, calc_premium

def load_customers(path: str) -> list[Customer]:
    customers = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try:
                customers.append(Customer(
                    name=row["name"],
                    age=int(row["age"]),
                    gender=row["gender"],
                    product=row["product"],
                    coverage=int(row["coverage"])
                ))
            except (KeyError, ValueError) as e:
                print("[경고] 건너뜀:", row, e)
    return customers


In [ ]:
def by_age(cs):
    groups = {
        "20대": [], "30대": [],
        "40대": [], "50대+": []
    }
    for c in cs:
        if c.age < 30:   k = "20대"
        elif c.age < 40: k = "30대"
        elif c.age < 50: k = "40대"
        else:            k = "50대+"
        groups[k].append(
            calc_premium(c)
        )
    return {
        k: round(sum(v)/len(v), 1)
        for k, v in groups.items() if v
    }


In [ ]:
# CSV 파일에서 고객 목록 읽기
customers = load_customers("data/customers.csv")
result = by_age(customers)

# 결과 출력
print(result)